### Match Schedule Scraping

In [1]:
import streamlit as st
import pandas as pd
import soccerdata as sd
import numpy as np
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from PIL import Image 
import requests
import os
import json
import time
import csv 

print("Packages imported successfully")

[10/10/26 15:56:21] INFO     No custom team name replacements found. You can configure these in       _config.py:91
                             C:\Users\rumja\soccerdata\config\teamname_replacements.json.                          

[10/10/26 15:56:22] INFO     No custom league dict found. You can configure additional leagues in    _config.py:189
                             C:\Users\rumja\soccerdata\config\league_dict.json.                                    

Packages imported successfully


In [2]:
raw_file = "data/raw/world_cup_matches_raw.csv"

In [3]:
ws = sd.WhoScored(
    leagues="INT-World Cup",
    seasons=2026
)

[10/10/26 15:57:18] INFO     Saving cached data to C:\Users\rumja\soccerdata\data\WhoScored          _common.py:250

In [4]:
def get_matches(stage_id, year, month):
    url = f"https://www.whoscored.com/tournaments/{stage_id}/data/?d={year}{month:02d}"

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    return pd.DataFrame(data['tournaments'][0]['matches'])                  

In [5]:
stages = ws.read_season_stages()
stage_ids = stages['stage_id'].unique().tolist()
stage_ids = [int(stage_id) for stage_id in stage_ids]

In [6]:
if os.path.exists(raw_file):
    world_cup_matches = pd.read_csv(raw_file)

else:
    all_matches = []

    for stage_id in stage_ids:
        if stage_id == 23752:
            all_matches.append(get_matches(stage_id, 2026, 6))
            all_matches.append(get_matches(stage_id, 2026, 7))
        else:
            all_matches.append(get_matches(stage_id, 2026, 6))

    world_cup_matches = pd.concat(all_matches, ignore_index=True)
    world_cup_matches.to_csv(raw_file, index=False)

### Event Scraping

In [7]:
match_ids = world_cup_matches['id'].unique().tolist()
match_ids

[1988523,
 1990729,
 1990970,
 1990731,
 1990967,
 1990969,
 1992061,
 1992064,
 1992065,
 1990730,
 1992067,
 1992063,
 1992066,
 1990971,
 1990968,
 1992062,
 1993367,
 1994739,
 1993896,
 1995347,
 1998192,
 1995348,
 1998571,
 1998572,
 1998582,
 1998994,
 1998901,
 1999238,
 2002449,
 2005991,
 2007643,
 2007644,
 1953853,
 1976987,
 1976985,
 1953854,
 1953855,
 1976986,
 1976989,
 1953856,
 1976988,
 1953857,
 1953858,
 1976990,
 1953860,
 1953859,
 1953861,
 1953862,
 1953863,
 1953864,
 1953865,
 1976983,
 1953866,
 1976984,
 1953867,
 1976982,
 1953868,
 1953869,
 1953871,
 1953870,
 1953872,
 1953873,
 1953892,
 1953891,
 1953894,
 1953893,
 1953895,
 1953896,
 1953874,
 1976979,
 1976980,
 1953906,
 1976981,
 1953875,
 1953876,
 1953877,
 1953879,
 1953878,
 1953880,
 1953881,
 1953882,
 1953883,
 1953884,
 1953885,
 1953886,
 1953887,
 1953888,
 1976997,
 1976995,
 1953889,
 1953890,
 1976996,
 1976991,
 1953897,
 1953898,
 1976992,
 1976993,
 1953899,
 1953900,
 1953901,


### Event Scraping 

In [8]:

from pathlib import Path

output_dir = Path("data/raw/world_cup_events_external")
output_dir.mkdir(parents=True, exist_ok=True)

expected_files = 104
csv_files = list(output_dir.glob("*.csv"))

if len(csv_files) == expected_files:
    print(f"Found all {expected_files} files")
else: 
    print(f"Found {len(csv_files)} of {expected_files}")


Found all 104 files


In [11]:
events_dir = Path("data/raw/world_cup_events_external")
master_path = Path("data/processed/world_cup_events_master.csv")

if master_path.exists():
    print("Existing dataset found")

else:
    csv_files = sorted(events_dir.glob("*.csv"))
    print("Files found:", len(csv_files))

    if len(csv_files) != 104:
        raise RuntimeError(
            f"Expected 104 files, found {len(csv_files)}. "
            "Check source files before rebuild"
        )
    
    dataframes = []
    failed_files = []
    
    for file in csv_files:
        try:
            df = pd.read_csv(file, low_memory=False)
            df["source_file"] = file.name
            dataframes.append(df)
        except Exception as error:
            failed_files.append((file.name, str(error)))
        
        if failed_files:
            print("\nFiles that failed:")
            for name, error in failed_files:
                print(name, "|", error)
            raise RuntimeError("Not rebuilt.")
            
        events_master = pd.concat(
            dataframes, ignore_index=True, sort=False
            )
            
        master_path.parent.mkdir(parentS=True, exist_ok=True)
        events_master.to_csv(master_path, index=False)
        
        print("Master dataset rebuilt:", master_path)
        print("Rows:", f"{len(events_master):,}")

Existing dataset found


In [13]:
import pandas as pd
from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

matches_path = processed_dir / "world_cup_matches.csv"
ids_path = processed_dir / "world_cup_match_ids.csv"

if matches_path.exists() and ids_path.exists():
    print("ID and match files exist")
else:
    if not matches_path.exists():
        world_cup_matches.to_csv(matches_path, index=False)
        print("Saved:", matches_path)
        
    if not ids_path.exists():
        pd.DataFrame(
            {"match_id": list(match_ids)}
        ).to_csv(ids_path, index=False)
        print("Saved:", ids_path)

Saved: data\processed\world_cup_matches.csv
Saved: data\processed\world_cup_match_ids.csv
